In [7]:
import pandas as pd
import requests
import json
from dotenv import load_dotenv
import os
import pandas as pd
import time
from time import sleep
from itertools import islice

load_dotenv()
API_KEY = os.getenv("API_KEY")
df1 = pd.read_csv('authors_data.csv')
df2 = pd.read_csv('works_ids.csv')
df3 = pd.read_csv('works_titles_abstracts.csv')
df4 = pd.read_csv('co_authors_data.csv')

In [8]:


indexes = df4.index[
    (df4["works_count"] >= 5) &
    (df4["works_count"] <= 5000)
].tolist()

subset = df4.loc[indexes, "author_id"]
subset

0       https://openalex.org/A5073831202
1       https://openalex.org/A5002330321
2       https://openalex.org/A5056499434
3       https://openalex.org/A5072230333
4       https://openalex.org/A5082591225
                      ...               
9769    https://openalex.org/A5075431039
9770    https://openalex.org/A5004566032
9771    https://openalex.org/A5037745332
9772    https://openalex.org/A5081271375
9773    https://openalex.org/A5075426054
Name: author_id, Length: 9557, dtype: object

In [9]:
import requests
import pandas as pd
from itertools import islice
from time import sleep

rows5 = []
rows6 = []

# Concept filters
css_concepts = ["C157449823", "C77805123", "C162324750", "C17744445"]
quant_concepts = ["C33923547", "C121332964", "C41008148"]

concept_filter1 = f"concept.id:{'|'.join(css_concepts)}"
concept_filter2 = f"concept.id:{'|'.join(quant_concepts)}"

BASE_URL = "https://api.openalex.org/works"

def chunked(iterable, size):
    it = iter(iterable)
    while True:
        chunk = list(islice(it, size))
        if not chunk:
            break
        yield chunk

# Keep track of all seen works to avoid duplicates
seen_work_ids = set()

# Make a set of valid author IDs: df1 authors + df4 co-authors
valid_author_ids = set(df1["author_id"].values) | set(df4["author_id"].values)

for batch_num, author_batch in enumerate(chunked(subset, 25), start=1):

    author_filter = "author.id:" + "|".join(author_batch)
    filter_string = (
        f"{author_filter},"
        f"cited_by_count:>10,"
        f"authors_count:<10,"
        f"{concept_filter1},"
        f"{concept_filter2}"
    )

    cursor = "*"
    print(f"\nProcessing batch {batch_num}")

    while cursor:
        params = {
            "filter": filter_string,
            "select": "publication_year,id,cited_by_count,authorships,title,abstract_inverted_index",
            "per-page": 200,
            "cursor": cursor,
            "api_key": API_KEY
        }

        try:
            response = requests.get(BASE_URL, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
        except requests.exceptions.RequestException as e:
            print(f"Batch {batch_num} failed: {e}")
            break

        results = data.get("results", [])
        if not results:
            break

        for item in results:
            work_id = item.get("id")
            if work_id in df2['id'].values or work_id in seen_work_ids:
                continue
            seen_work_ids.add(work_id)

            # Extract all author IDs from authorships
            authorships = item.get("authorships", [])
            all_author_ids = [auth["author"]["id"] for auth in authorships if auth.get("author")]

            # Keep only IDs in df1 or df4
            filtered_author_ids = [aid for aid in all_author_ids if aid in valid_author_ids]

            rows5.append({
                "id": work_id,
                "publication_year": item.get("publication_year"),
                "cited_by_count": item.get("cited_by_count"),
                "author_ids": filtered_author_ids
            })

            rows6.append({
                "id": work_id,
                "title": item.get("title"),
                "abstract_index": item.get("abstract_inverted_index", {})
            })

        cursor = data.get("meta", {}).get("next_cursor")
        sleep(0.1)

    sleep(0.5)

df5 = pd.DataFrame(rows5)
df6 = pd.DataFrame(rows6)

print("\nComplete!")
print(len(df5), len(df6))


Processing batch 1

Processing batch 2

Processing batch 3

Processing batch 4

Processing batch 5

Processing batch 6

Processing batch 7

Processing batch 8

Processing batch 9

Processing batch 10

Processing batch 11

Processing batch 12

Processing batch 13

Processing batch 14

Processing batch 15

Processing batch 16

Processing batch 17

Processing batch 18

Processing batch 19

Processing batch 20

Processing batch 21

Processing batch 22

Processing batch 23

Processing batch 24

Processing batch 25

Processing batch 26

Processing batch 27

Processing batch 28

Processing batch 29

Processing batch 30

Processing batch 31

Processing batch 32

Processing batch 33

Processing batch 34

Processing batch 35

Processing batch 36

Processing batch 37

Processing batch 38

Processing batch 39

Processing batch 40

Processing batch 41

Processing batch 42

Processing batch 43

Processing batch 44

Processing batch 45

Processing batch 46

Processing batch 47

Processing batch 48



In [10]:
df5.to_csv("co_authors_works.csv",index = False)
df6.to_csv("co_authors_works_titles_abstracts.csv",index = False)
